In [36]:
# 1. Libraries
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from detector_io import load_detector_image

# Choose the detector before running the notebook.
detector = "CMOS"                # "CMOS" or "MCP"

def load_image(index):
    return load_detector_image(index, detector, input_folder, image_key)

In [ ]:
# 2. Choose image, dark scan, and display limits
input_folder = Path("raw") / detector
output_folder = Path("output") / detector.lower()
image_index = 420 if detector == "CMOS" else 0
dark_scan_index = 416 if detector == "CMOS" else None
image_key = "data_22"
percentiles = [1, 90]           # display limits outside the beamstop
mask_path = output_folder / "beamstop_mask.png"  # None to disable

# MCP integers are zero-based indices; filenames are also accepted by load_image.
# Set scan lists/fields for your MCP acquisition before analyzing a series.
if detector == "MCP":
    for index, path in enumerate(sorted(input_folder.glob("*_TDC_2D_*.txt"))):
        print(index, path.name)


In [ ]:
# 3. Load one image and optionally subtract the dark
image = load_image(image_index)
if dark_scan_index is not None:
    image = image - load_image(dark_scan_index)
print("Corrected intensity range:", np.nanmin(image), np.nanmax(image))

# Paint mask: red pixels are excluded only when choosing the color limits.
mask = np.zeros(image.shape, dtype=bool)
if mask_path is not None:
    with Image.open(mask_path) as file:
        rgba = np.asarray(file.convert("RGBA"))
    mask = np.flipud((rgba[..., 0] >= 200) & (rgba[..., 1] < 100) &
                     (rgba[..., 2] < 100) & (rgba[..., 3] > 0))
mi, ma = np.nanpercentile(image[~mask & np.isfinite(image)], percentiles)
print(f"Percentile color limits outside beamstop: {mi:g}, {ma:g}")

In [ ]:
# 4. Display the corrected image with a colorbar
fig, ax = plt.subplots(figsize=(9, 8), constrained_layout=True)
artist = ax.imshow(image, origin="lower", cmap="magma", vmin=mi, vmax=ma)
ax.set(title=f"{detector} image {image_index} | dark {dark_scan_index}",
       xlabel="Column (px)", ylabel="Row (px)")
fig.colorbar(artist, ax=ax, label="Dark-subtracted intensity")
plt.show()

In [ ]:
# 5. Save the figure exactly as plotted, plus numerical pixel values
output_folder.mkdir(parents=True, exist_ok=True)
name = f"scan_{image_index:04d}" + (f"_dark_{dark_scan_index:04d}" if dark_scan_index is not None else "")
np.save(output_folder / f"{name}.npy", image)  # numerical values, including negatives
fig.savefig(output_folder / f"{name}.png", dpi=150)
# PNG includes the plotted colormap, limits, axes, title, and colorbar.
print("Saved:", output_folder / name)